### Imports

In [1]:
import pandas as pd
import os
from testgen.utils import *
from pathlib import Path
from dotenv import load_dotenv

In [2]:
base_path = Path().cwd()
load_dotenv(override=True)

True

In [3]:
# read data and rename columns
df = pd.read_excel(base_path / "data/sensor_requirements.xlsx")
df_examples = pd.read_excel(base_path / "data/sensor_examples.xlsx")

# Find Examples

In [4]:
N_EXAMPLES = 1

examples_txt, examples = get_example_txt(df_examples, N_EXAMPLES)

print("\n".join(examples_txt.split("\n")[-10:]))

Requirement: Drift control systems must adaptively use yaw rate feedback to optimize vehicle control during high-performance driving scenarios
Target Sensor/s: [yaw_rate]

Requirement: The power steering system must adapt the torque levels in response to detected road surface conditions (e.g., ice, water, gravel)
Target Sensor/s: [steering_torque]

Requirement: The control system must detect and mitigate power oversteer to maintain vehicle control by adjusting the wheel steering angle and acceleration pedal inputs
Target Sensor/s: [acceleration_pedal, wheel_steering_angle]




# Response Format

In [5]:
from testgen.prompts.Sensors import Sensors
from typing import List
from pydantic import BaseModel, Field, create_model

Sensors_t = Sensors.split("\n")


def clean_sensors_fn(x):
    x = x.split(":")
    x[0] = x[0][: x[0].find("(")]
    return x


Sensors_t = list(map(clean_sensors_fn, Sensors_t))

sensor_attrs = {}

for sensor in Sensors_t:
    sensor_attrs[sensor[0].strip().lower().replace(" ", "_")] = (
        int,
        Field(description=sensor[1].strip()),
    )


TargetSensor = create_model("TargetSensor", **sensor_attrs)


class TargetSensorList(BaseModel):
    req_id: int = Field(description="Reauirement's ID")
    text: str = Field(description="Reauirement's text")
    target_sensor: TargetSensor = Field(
        description="List of sensors where 1 is the targeted sensor and 0 is not"
    )


class RequirementList(BaseModel):
    requirements: List[TargetSensorList] = Field(
        description="List of requirements and their TargetSensorList"
    )

In [6]:
# print(json.dumps(TargetSensor.model_json_schema(), indent=2))

In [7]:
# print(json.dumps(TargetSensorList.model_json_schema(), indent=2))

# LLM

In [8]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPromptBulk

In [9]:
llm_models = {
    "azure": ["gpt-4o-mini"],
    # "azure": ["gpt-4o-mini", "gpt-4o"],
    # "novita": [
    #     "qwen/qwen2.5-7b-instruct",
    #     "google/gemma-3-27b-it",
    #     "meta-llama/llama-3-70b-instruct",
    # ],
}

endpoint_attrs = {
    "azure": {
        "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
        "api_version": os.getenv("AZURE_API_VERSION"),
        "base_url": os.getenv("AZURE_OPENAI_ENDPOINT"),
    },
    "novita": {
        "api_key": os.getenv("NOVITA_API_KEY"),
        "base_url": os.getenv("NOVITA_ENDPOINT"),
    },
}

### Get Requirements for multi prediction

In [11]:
N_REQS = 2
SAMPLE_TYPE = "random"

batches = get_batches(df, SAMPLE_TYPE, N_REQS)
batches, req_texts = requirement_text_bulk(batches, df.columns)

Number of Batches: 48
Number of Instances left: 1


In [12]:
print(req_texts[0])

<ID> 63 <Text> The vehicle must have a throttle override mechanism that prioritizes braking sensing input over the accelerattion sensing  when both are pressed simultaneously.
<ID> 78 <Text> The vehicle's steering  must be consistent with the driver's torque sensing on the steering wheel, with no uncommanded delays or discrepancies.



In [ ]:
for endpoint_name in llm_models.keys():

    for model_name in llm_models[endpoint_name]:

        print(
            f"Running {model_name} on {endpoint_name} examples {N_EXAMPLES} batch_size {N_REQS}..."
        )

        client = llm_client(endpoint_name, **endpoint_attrs[endpoint_name])

        results = invoke_bulk_sensor(
            endpoint_name,
            model_name,
            client,
            batches,
            req_texts,
            examples_txt,
            SystemPrompt,
            Sensors,
            UserPromptBulk,
            response_format=RequirementList,
        )

        (
            number_of_requests,
            accuracy,
            avg_time_per_req,
            avg_token_per_req,
            avg_completion_token_per_req,
            total_tokens,
            total_completion_tokens,
            total_time,
        ) = calc_stats(results, number_of_requests=len(batches))

        results_file = save_responses(
            base_path,
            "bulk",
            model_name=model_name,
            n_examples=N_EXAMPLES,
            batch_size=N_REQS,
            examples=examples,
            accuracy=accuracy,
            number_of_requests=number_of_requests,
            total_tokens=total_tokens,
            total_completion_tokens=total_completion_tokens,
            avg_token_per_req=avg_token_per_req,
            avg_completion_token_per_req=avg_completion_token_per_req,
            avg_time_per_req=avg_time_per_req,
            results=results,
        )

Running gpt-4o-mini on azure examples 1 batch_size 2...


  0%|          | 0/48 [00:00<?, ?it/s]